In [7]:
import sys
sys.path.append("/mnt/workspace/gzchen/train")
from lightseq.lightseq_async_attn import _lightseq_forward
import torch

dtype = torch.bfloat16
batch_size = 1
seqlen = 4*1024
nheads = 32
d = 128
dropout_p = 0
causal = True
deterministic = False

device = "cuda:0"

# assert causal
# assert seqlen % (2 * world_size) == 0
# assert d % 8 == 0

# qkv = torch.randn(
#     batch_size, seqlen, 3, nheads, d, device=device, dtype=dtype, requires_grad=True
# )
print(device)
num_kv_heds = 8
q = torch.randn(
    batch_size, seqlen, nheads, d, device=device, dtype=dtype, requires_grad=True
)
k = torch.randn(
    batch_size, seqlen, num_kv_heds, d, device=device, dtype=dtype, requires_grad=True
)
v = torch.randn(
    batch_size, seqlen, num_kv_heds, d, device=device, dtype=dtype, requires_grad=True
)

cuda:0


In [17]:
from lightseq.async_communication import initialize_distributed
import os
os.environ["RANK"] = '0'
initialize_distributed()

_lightseq_forward(q..transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2), True, 0.5, "lightseq")

SyntaxError: invalid syntax (1991604230.py, line 6)

In [15]:
import torch.distributed as dist
class _attention(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, causal, sm_scale):
        # try:
        #     global args
        #     comm_mode = args.comm_mode
        #     backward_engine = args.backward_engine
        # except:
        comm_mode = 'lightseq'
        backward_engine = 'flash'
        
        q, k, v, o, L = _lightseq_forward(q, k, v, causal, sm_scale, comm_mode)

        ctx.save_for_backward(q, k, v, o, L)
        ctx.sm_scale = sm_scale
        ctx.comm_mode = comm_mode
        ctx.backward_engine = backward_engine
        return o

    @staticmethod
    def backward(ctx, do):
        q, k, v, o, L = ctx.saved_tensors 
        sm_scale = ctx.sm_scale

        dq, dk, dv = _lightseq_backward(do, q, k, v, o, L, sm_scale, ctx.comm_mode, ctx.backward_engine)
        return dq, dk, dv, None, None

attention = _attention.apply


#@pytest.mark.parametrize('causal', [False, True])
#@pytest.mark.parametrize('Z, H, N_CTX, D_HEAD', [(6, 9, 1024, 64)])
def test_op(Z, H, N_CTX, D_HEAD, causal, dtype=torch.float16):
    torch.manual_seed(20)
    q = torch.empty((Z, H, N_CTX, D_HEAD), dtype=dtype, device="cuda").normal_(mean=0., std=0.5).requires_grad_()
    k = torch.empty((Z, H, N_CTX, D_HEAD), dtype=dtype, device="cuda").normal_(mean=0., std=0.5).requires_grad_()
    v = torch.empty((Z, H, N_CTX, D_HEAD), dtype=dtype, device="cuda").normal_(mean=0., std=0.5).requires_grad_()
    
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    seq_per_rank = N_CTX // world_size

    sm_scale = 0.5
    dout = torch.randn_like(q)
    # reference implementation
    M = torch.tril(torch.ones((N_CTX, N_CTX), device="cuda"))
    p = torch.matmul(q, k.transpose(2, 3)) * sm_scale
    assert causal
    if causal:
        p[:, :, M == 0] = float("-inf")
    p = torch.softmax(p.float(), dim=-1).half()
    ref_out = torch.matmul(p, v)
    ref_out.backward(dout)
    ref_dv, v.grad = v.grad.clone(), None
    ref_dk, k.grad = k.grad.clone(), None
    ref_dq, q.grad = q.grad.clone(), None

    # triton implementation
   
    a, b, c, d = q.size()
    real_q = q[:,:, rank * seq_per_rank: (rank + 1) * seq_per_rank, :].view(a, b, -1, d).contiguous().clone().detach().requires_grad_(True)
    real_k = k[:,:, rank * seq_per_rank: (rank + 1) * seq_per_rank, :].view(a, b, -1, d).contiguous().clone().detach().requires_grad_(True)
    real_v = v[:,:, rank * seq_per_rank: (rank + 1) * seq_per_rank, :].view(a, b, -1, d).contiguous().clone().detach().requires_grad_(True)
    real_do = dout[:,:, rank * seq_per_rank: (rank + 1) * seq_per_rank, :].view(a, b, -1, d).contiguous().clone().detach().requires_grad_(True)
    
    tri_out = attention(real_q, real_k, real_v, causal, sm_scale).half()

    # compare
    assert torch.allclose(ref_out[:, :, rank * seq_per_rank: (rank + 1) * seq_per_rank, :], tri_out, atol=1e-2, rtol=0), f" rank {rank} fails forward"
    print(f" *** rank {rank} passes forward")
    tri_out.backward(real_do)
    tri_dv, real_v.grad = real_v.grad.clone(), None
    tri_dk, real_k.grad = real_k.grad.clone(), None
    tri_dq, real_q.grad = real_q.grad.clone(), None
    assert torch.allclose(ref_dq[:, :, rank * seq_per_rank: (rank + 1) * seq_per_rank, :], tri_dq, atol=1e-2, rtol=0),  f" rank {rank} fails backward dq"
    assert torch.allclose(ref_dk[:, :, rank * seq_per_rank: (rank + 1) * seq_per_rank, :], tri_dk, atol=1e-2, rtol=0),  f"rank {rank} fails backward dk" #f" {ref_dk[:, :, rank * seq_per_rank: (rank + 1) * seq_per_rank, :]} {tri_dk} {torch.max(ref_dk[:, :, rank * seq_per_rank: (rank + 1) * seq_per_rank, :] - tri_dk)} rank {rank} fails backward dk"
    assert torch.allclose(ref_dv[:, :, rank * seq_per_rank: (rank + 1) * seq_per_rank, :], tri_dv, atol=1e-2, rtol=0),  f"rank {rank} fails backward dv {ref_dv[:, :, rank * seq_per_rank: (rank + 1) * seq_per_rank, :]} {tri_dv} {torch.max(ref_dv[:, :, rank * seq_per_rank: (rank + 1) * seq_per_rank, :] - tri_dv)} rank {rank} fails backward dv"
    print(f"rank {rank} passes backward")

In [16]:
test_op(1, 32, 4096, 128, True)

RuntimeError: Default process group has not been initialized, please make sure to call init_process_group.